# EUDR DeepLabV3 Training — Kaggle GPU

**GPU:** P100 (recommended) or T4 x2  
**Runtime:** 30 hrs/week quota — enable via *Settings → Accelerator*

### Before running:
Add these Kaggle Secrets (*Settings → Secrets*):
- `CDSE_USER` — Copernicus Data Space username
- `CDSE_PASSWORD` — Copernicus Data Space password
- `GEE_PROJECT_ID` — Google Earth Engine project ID
- `GEE_SERVICE_ACCOUNT_KEY` — GEE service account JSON (base64 encoded)

In [ ]:
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | "
          f"{torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

In [ ]:
# Clone repo and install dependencies
import subprocess, os

REPO_URL = "https://github.com/rajul-kk/EUDR-compliance.git"  # update if needed
WORK_DIR = "/kaggle/working/eudr"

if not os.path.exists(WORK_DIR):
    subprocess.run(["git", "clone", REPO_URL, WORK_DIR], check=True)
else:
    subprocess.run(["git", "-C", WORK_DIR, "pull"], check=True)

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install Python dependencies
!pip install -q -r requirements.txt

In [ ]:
# Load credentials from Kaggle Secrets into environment
import base64, json, os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

os.environ["CDSE_USER"]      = secrets.get_secret("CDSE_USER")
os.environ["CDSE_PASSWORD"]  = secrets.get_secret("CDSE_PASSWORD")
os.environ["GEE_PROJECT_ID"] = secrets.get_secret("GEE_PROJECT_ID")

# Write GEE service account key to disk
gee_key_b64 = secrets.get_secret("GEE_SERVICE_ACCOUNT_KEY")
gee_key_path = "/kaggle/working/gee_key.json"
with open(gee_key_path, "w") as f:
    f.write(base64.b64decode(gee_key_b64).decode())
os.environ["GEE_SERVICE_ACCOUNT_KEY_PATH"] = gee_key_path

print("Credentials loaded.")

In [ ]:
# Create required directories
dirs = [
    "inputs", "models", "reports", "reports/dds", "reports/vectors",
    "data/raw_satellite/2020_baseline",
    "data/raw_satellite/2024_current",
    "data/hybrid_masks",
    "data/predictions_2024_deeplab",
]
for d in dirs:
    os.makedirs(d, exist_ok=True)
print("Directories ready.")

In [ ]:
# Step 1: Discover farms from OpenStreetMap
# Skip if farms_osm.csv already exists (e.g. from a Kaggle Dataset attachment)
if not os.path.exists("inputs/farms_osm.csv") or os.path.getsize("inputs/farms_osm.csv") < 100:
    !python src/find_farms.py
else:
    import pandas as pd
    n = len(pd.read_csv("inputs/farms_osm.csv"))
    print(f"Using existing farms_osm.csv ({n} farms)")

In [ ]:
# Step 2: Download Sentinel-2 cloud-free composites
!python src/sentinel_client.py \
    --csv-path inputs/farms_osm.csv \
    --year 2020 \
    --output-dir data/raw_satellite/2020_baseline

!python src/sentinel_client.py \
    --csv-path inputs/farms_osm.csv \
    --year 2024 \
    --output-dir data/raw_satellite/2024_current

In [ ]:
# Step 3: Generate hybrid masks from GEE (DynamicWorld + Canopy Height)
!python GEE_dynamic/src/generate_hybrid.py

In [ ]:
# Step 4: Train DeepLabV3 with AMP + DataParallel
# AMP uses P100 FP16; DataParallel splits batches across both T4s if available
!python src/ML_farm_net.py \
    --raw-dir data/raw_satellite/2020_baseline \
    --mask-dir data/hybrid_masks \
    --output-model-path models/farm_deeplab.pth \
    --epochs 10 \
    --batch-size 8 \
    --learning-rate 1e-4 \
    --seed 42

In [ ]:
# Step 5: Run inference on 2024 imagery
!python src/inference.py \
    --model-path models/farm_deeplab.pth \
    --input-dir data/raw_satellite/2024_current \
    --output-dir data/predictions_2024_deeplab \
    --model-type deeplab

In [ ]:
# Step 6: Detect deforestation + generate report
import os
env = os.environ.copy()
env["EUDR_PRED_DIR"] = "data/predictions_2024_deeplab"

import subprocess
subprocess.run(["python", "src/detect_deforestation.py"], env=env, check=True)

In [ ]:
# Step 7: Write audit trail
from src.logging_config import setup_pipeline_logging
setup_pipeline_logging()

import json
from src.audit_trail import AuditLog, build_audit_entry

summary = {}
if os.path.exists("reports/summary_stats.json"):
    with open("reports/summary_stats.json") as f:
        summary = json.load(f)

entry = build_audit_entry(
    model_type="deeplab",
    model_version="kaggle-run",
    input_image_dir="data/raw_satellite/2024_current",
    prediction_dir="data/predictions_2024_deeplab",
    report_csv_path="reports/deforestation_report.csv",
    summary=summary,
)
log = AuditLog(log_path="reports/audit_log.jsonl")
entry = log.append(entry)
print(f"Audit entry written (run_id={entry.run_id})")

In [ ]:
# Preview results
import pandas as pd

report = pd.read_csv("reports/deforestation_report.csv")
print(f"Total farms assessed: {len(report)}")
print(report["alert_level"].value_counts().to_string())
print()
print(report[["farm_id", "deforestation_percent", "alert_level"]].head(10).to_string(index=False))

In [ ]:
# Package outputs for download
!zip -r /kaggle/working/eudr_outputs.zip \
    models/farm_deeplab.pth \
    reports/deforestation_report.csv \
    reports/summary_stats.json \
    reports/audit_log.jsonl

print("Download eudr_outputs.zip from the Output panel.")